
Text summarization using NLP

Last Updated: April 8th, 2025

Daily Challenge : Text summarization using NLP


Introduction

This notebook demonstrates a practical application of Natural Language Processing (NLP) techniques to automatically generate summaries of text documents. We will explore how to preprocess text data, represent words and sentences as vectors, and leverage graph-based algorithms to identify the most important sentences for summarization.


👩‍🏫 👩🏿‍🏫 What You’ll learn

    Text Preprocessing: Techniques for cleaning and preparing text data, including tokenization, stop word removal, and converting text to lowercase.
    Word Embeddings: Understanding and using pre-trained word embeddings like GloVe to represent words as dense vectors.
    Sentence Vectorization: Creating vector representations of sentences by aggregating word embeddings.
    Similarity Measures: Using cosine similarity to determine the semantic similarity between sentences.
    Graph-Based Summarization: Applying the PageRank algorithm on a graph of sentence similarities to rank sentence importance.
    Text Summarization Implementation: Combining these techniques to build a text summarization system.


🛠️ What you will create

You will create an automatic text summarization system that can take a collection of tennis articles as input and generate a concise summary highlighting the key information.


Task

1. Data Loading and Inspection

    Load the tennis articles dataset from the .xls file using pandas.
    Explore the dataset using .head() and .info() to understand its structure.
    Drop the article_title column to simplify the dataset.

2. Sentence Tokenization

    Use nltk.sent_tokenize() to split the article_text into individual sentences.
    Flatten the resulting list of sentence lists into a single list of all sentences.

3. Download and Load GloVe Word Embeddings

    Download the pre-trained GloVe vectors (e.g., glove.6B.100d.txt).
    Load the embeddings into a Python dictionary where each word maps to its 100-dimensional vector.

4. Text Cleaning and Normalization

    Remove punctuation, special characters, and numbers using regex.
    Convert all sentences to lowercase to avoid case-sensitive mismatch.
    Remove stop words using nltk.corpus.stopwords to reduce noise in the data.

5. Sentence Vectorization

    For each cleaned sentence:
        Split into words.
        Replace each word with its GloVe vector (use a zero-vector if the word is not in the embedding).
        Compute the average of all word vectors in the sentence.
    Store all resulting sentence vectors in a list.

6. Similarity Matrix Construction

    Initialize an empty matrix of size (number of sentences × number of sentences).
    Compute pairwise cosine similarity between sentence vectors.
    Fill in the matrix such that each cell represents the similarity between two sentences.

7. Graph Construction and Sentence Ranking

    Convert the similarity matrix into a graph using networkx.
    Apply the PageRank algorithm to score the importance of each sentence.

8. Summarization

    Sort all sentences based on their PageRank scores in descending order.
    Extract the top N sentences (e.g., 10) as the final summary.
    Print or return the summarized sentences.


In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
import nltk
import networkx as nx
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

In [5]:

CSV_PATH   = "tennis_articles.csv"
GLOVE_PATH = "glove.6B.100d.txt"
TOP_N      = 10
EMB_DIM    = 100


# Chargement robuste du CSV (auto-détection du séparateur et skip des lignes corrompues)
def robust_read_csv(path):
    try:
        return pd.read_csv(path, sep=None, engine="python", on_bad_lines="skip")
    except Exception:
        # Essais communs
        for sep in [",", ";", "\t", "|"]:
            try:
                return pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")
            except Exception:
                pass
        raise

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found at {CSV_PATH}")

pdf = pd.read_csv(
    CSV_PATH,
    sep=None,              # laisse pandas deviner le séparateur
    engine="python",
    on_bad_lines="skip",
    encoding="latin-1",    # <-- clé du problème
)

print("Columns found:", list(pdf.columns))
print("Shape:", pdf.shape)
print(pdf.head())

# On s’attend à une colonne 'article_title' et 'article_text' d’après l’énoncé,
# mais on rend le script tolérant si les noms diffèrent.
CANDIDATE_TEXT_COLS = ["article_text", "text", "content", "article", "body"]
text_col = None
for c in CANDIDATE_TEXT_COLS:
    if c in pdf.columns:
        text_col = c
        break

if text_col is None:
    raise ValueError(
        f"Aucune colonne texte standard trouvée. Colonnes présentes : {list(pdf.columns)}.\n"
        f"Ajoutez votre colonne de texte à CANDIDATE_TEXT_COLS."
    )

if "article_title" in pdf.columns:
    pdf = pdf.drop(columns=["article_title"])


nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

sent_tokenize = nltk.sent_tokenize

# Split en phrases pour chaque article, puis flatten
articles = pdf[text_col].astype(str).tolist()
all_sentences = []
for art in articles:
    sents = sent_tokenize(art)
    all_sentences.extend(sents)

print(f"\nNombre total de phrases : {len(all_sentences)}")


if not os.path.exists(GLOVE_PATH):
    raise FileNotFoundError(
        f"GloVe non trouvé à {GLOVE_PATH}. Téléchargez glove.6B.100d.txt et mettez à jour GLOVE_PATH."
    )

def load_glove(path, dim=100):
    embeddings = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading GloVe"):
            parts = line.strip().split()
            word = parts[0]
            vec = np.asarray(parts[1:], dtype="float32")
            if vec.shape[0] == dim:
                embeddings[word] = vec
    return embeddings

glove = load_glove(GLOVE_PATH, dim=EMB_DIM)
print(f"GloVe loaded: {len(glove):,} words.")


stopwords = set(nltk.corpus.stopwords.words("english"))
def clean_sentence(s: str) -> str:
    # remove non-letters and numbers
    s = re.sub(r"[^a-zA-Z\s]", " ", s)
    s = s.lower().strip()
    # remove multiple spaces
    s = re.sub(r"\s+", " ", s)
    # remove stopwords
    tokens = [w for w in s.split() if w not in stopwords]
    return " ".join(tokens)

cleaned_sentences = [clean_sentence(s) for s in all_sentences]


def sentence_vector(sent: str, emb_dict, dim=100):
    if not sent:
        return np.zeros(dim, dtype="float32")
    words = sent.split()
    vecs = [emb_dict.get(w, np.zeros(dim, dtype="float32")) for w in words]
    if len(vecs) == 0:
        return np.zeros(dim, dtype="float32")
    return np.mean(vecs, axis=0)

sentence_vectors = np.vstack([sentence_vector(s, glove, EMB_DIM) for s in cleaned_sentences])
print("Sentence vectors shape:", sentence_vectors.shape)

if len(sentence_vectors) == 0:
    raise ValueError("Aucune phrase après nettoyage. Vérifiez le csv ou les colonnes.")

sim_matrix = cosine_similarity(sentence_vectors)
# On met la diagonale à 0 pour éviter de favoriser les phrases identiques à elles-mêmes
np.fill_diagonal(sim_matrix, 0.0)


graph = nx.from_numpy_array(sim_matrix)
scores = nx.pagerank(graph, max_iter=100, tol=1e-6)

ranked_sentences_idx = sorted(scores, key=scores.get, reverse=True)
top_idx = ranked_sentences_idx[:TOP_N]

# On garde l’ordre original des phrases pour plus de lisibilité du résumé
top_idx_sorted = sorted(top_idx)
summary_sentences = [all_sentences[i] for i in top_idx_sorted]

print("\n" + "="*80)
print(f"Résumé (Top {TOP_N} phrases) :\n")
for s in summary_sentences:
    print("-", s)
print("="*80)

Columns found: ['article_id', 'article_title', 'article_text', 'source']
Shape: (8, 4)
   article_id                                      article_title  \
0           1  I do not have friends in tennis, says Maria Sh...   
1           2  Federer defeats Medvedev to advance to 14th Sw...   
2           3  Tennis: Roger Federer ignored deadline set by ...   
3           4  Nishikori to face off against Anderson in Vien...   
4           5  Roger Federer has made this huge change to ten...   

                                        article_text  \
0  Maria Sharapova has basically no friends as te...   
1  BASEL, Switzerland (AP)  Roger Federer advanc...   
2  Roger Federer has revealed that organisers of ...   
3  Kei Nishikori will try to end his long losing ...   
4  Federer, 37, first broke through on tour over ...   

                                              source  
0  https://www.tennisworldusa.org/tennis/news/Mar...  
1  http://www.tennis.com/pro-game/2018/10/copil-s...  
2 

Loading GloVe: 400000it [00:03, 105908.96it/s]

GloVe loaded: 400,000 words.
Sentence vectors shape: (130, 100)

Résumé (Top 10 phrases) :

- So I'm not the one to strike up a conversation about the weather and know that in the next few minutes I have to go and try to win a tennis match.
- Speaking at the Swiss Indoors tournament where he will play in Sundays final against Romanian qualifier Marius Copil, the world number three said that given the impossibly short time frame to make a decision, he opted out of any commitment.
- Major players feel that a big event in late November combined with one in January before the Australian Open will mean too much tennis and too little rest.
- Currently in ninth place, Nishikori with a win could move to within 125 points of the cut for the eight-man event in London next month.
- He used his first break point to close out the first set before going up 3-0 in the second and wrapping up the win on his first match point.
- I felt like the best weeks that I had to get to know players when I was p